# ДЗ 16. Pandas: первая таблица, столбцы и фильтры
**Киберформула · Python и анализ данных**

Файлы: `orders.csv` (чистая выгрузка заказов) и `orders_raw.csv` (та же выгрузка до чистки).

| Уровень | Задачи |
|---|---|
| **База** | 1–3 |
| **Челлендж** | 4–7 |
| **Про** | 8–9 |

**Правило:** после каждого ответа напишите одну фразу-вывод комментарием `# Вывод: ...`. Число без вывода — половина ответа.
Если застряли, откройте справочник урока 16.

In [3]:
# Загрузка файлов: в Colab нажмите «Выбрать файлы» и укажите orders.csv и orders_raw.csv
import os
need = [f for f in ["orders.csv", "orders_raw.csv"] if not os.path.exists(f)]
if need:
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и orders_raw.csv в одну папку с ноутбуком")
for f in ["orders.csv", "orders_raw.csv"]:
    print(f, "на месте:", os.path.exists(f))

import pandas as pd

orders.csv на месте: True
orders_raw.csv на месте: True


## База

### Задача 1. Первый взгляд
Загрузите `orders.csv` в переменную `df`.
1. Сколько в таблице строк и столбцов?
2. Сколько заказов из Караганды?
3. Какие товары продаются в категории «Аксессуары» и сколько их разных?

In [ ]:
df = pd.read_csv("orders.csv")

print("Размер (строки, столбцы):", df.shape)

krg = df[df["city"] == "Караганда"]
print("Заказов из Караганды:", len(krg))

acc = df[df["category"] == "Аксессуары"]["product"].unique()
print("Товары в «Аксессуарах»:", acc)
print("Разных товаров:", len(acc))
# Вывод: в таблице 500 заказов и 9 столбцов; из Караганды 72 заказа — это около 14% всех продаж; в «Аксессуарах» всего 4 разных товара (клавиатура, наушники, power bank, мышь), ассортимент узкий.

Размер (строки, столбцы): (500, 9)
Заказов из Караганды: 72
Товары в «Аксессуарах»: <StringArray>
['Клавиатура Logitech K120',    'Наушники JBL Tune 520',
  'Power bank Xiaomi 20000',       'Мышь Logitech M185']
Length: 4, dtype: str
Разных товаров: 4


### Задача 2. Дорогие заказы Шымкента
Добавьте столбец `revenue` = `price` × `qty`. Выведите 5 самых дорогих **оплаченных** заказов из Шымкента: только столбцы `date`, `product`, `qty`, `revenue`.

In [9]:
PAID = "оплачен"
CANCELLED = "отменён"

In [ ]:
df["revenue"] = df["price"] * df["qty"]
paid = df[df["status"] == PAID]

shym = paid[paid["city"] == "Шымкент"]
top5 = shym.sort_values("revenue", ascending=False).head(5)
print(top5[["date", "product", "qty", "revenue"]])
# Вывод: все 5 самых дорогих оплаченных заказов Шымкента — ноутбуки; первые два места заняли заказы по 2 штуки более дешёвых моделей (ASUS, Lenovo), а самый дорогой ноутбук MacBook Air M3 брали по одному — выручку дают и высокая цена, и количество.

           date           product  qty  revenue
202  2026-06-30  ASUS VivoBook 15    2   638000
162  2026-06-05  Lenovo IdeaPad 5    2   578000
62   2026-04-07    MacBook Air M3    1   549000
138  2026-05-26    MacBook Air M3    1   549000
84   2026-04-19    MacBook Air M3    1   549000


### Задача 3. Покупают помногу?
1. Сколько заказов, в которых купили **2 штуки или больше**? Какая это доля всех заказов в %?
2. Какой товар чаще всего покупают сразу **по 3 штуки**?

In [ ]:
many = df[df["qty"] >= 2]
print("Заказов с qty >= 2:", len(many))
print("Доля:", round(len(many) / len(df) * 100, 1), "%")

three = df[df["qty"] == 3]
print(three["product"].value_counts().head(3))
print("Чаще всего по 3 штуки:", three["product"].value_counts().idxmax())
# Вывод: 116 заказов (23.2%) — на 2 штуки и больше, значит, в 3 из 4 заказов берут одну вещь; по 3 штуки чаще всего покупают наушники JBL Tune 520 (13 раз), а сразу за ними мышь и клавиатура — помногу берут недорогие аксессуары, а не технику.

Заказов с qty >= 2: 116
Доля: 23.2 %
product
Наушники JBL Tune 520       13
Мышь Logitech M185          12
Клавиатура Logitech K120    11
Name: count, dtype: int64
Чаще всего по 3 штуки: Наушники JBL Tune 520


## Челлендж

### Задача 4. Алматы против Астаны
Для **оплаченных** заказов посчитайте средний чек (`revenue`) и медиану отдельно для Алматы и для Астаны.
Средние получатся почти одинаковые, а медианы — нет. Объясните, что это говорит о покупках в двух городах.

In [ ]:
for city in ["Алматы", "Астана"]:
    r = paid[paid["city"] == city]["revenue"]
    print(city, "| средний чек:", round(r.mean()), "| медиана:", round(r.median()))
# Вывод: средний чек в Алматы и Астане почти одинаковый (~187–188 тыс.), но медиана в Алматы ниже (129 тыс. против 159 тыс.) — в Алматы типичная покупка дешевле, а среднее поднимают редкие очень дорогие заказы; в Астане покупки ровнее и обычный чек выше.

Алматы | средний чек: 187184 | медиана: 129000
Астана | средний чек: 188080 | медиана: 159000


### Задача 5. Весна против лета
Сравните **весну** (март–май) и **лето** (июнь–август) по оплаченным заказам:
выручка, изменение в %, число заказов.

Подсказка: `month = df["date"].str[:7]` и `month.isin(["2026-03", "2026-04", "2026-05"])`.

In [ ]:
month = paid["date"].str[:7]
spring = paid[month.isin(["2026-03", "2026-04", "2026-05"])]
summer = paid[month.isin(["2026-06", "2026-07", "2026-08"])]

spring_rev = spring["revenue"].sum()
summer_rev = summer["revenue"].sum()
change = (summer_rev - spring_rev) / spring_rev * 100

print("Весна: выручка", spring_rev, "| заказов", len(spring))
print("Лето:  выручка", summer_rev, "| заказов", len(summer))
print("Изменение выручки:", round(change, 1), "%")
# Вывод: летом выручка выросла на 26.2% по сравнению с весной, а число заказов — на 32% (188 против 142); рост дали больше покупателей, а не дорогие покупки: средний чек летом даже немного снизился (~191 тыс. против ~200 тыс.).

Весна: выручка 28450500 | заказов 142
Лето:  выручка 35892500 | заказов 188
Изменение выручки: 26.2 %


### Задача 6. Где чаще отменяют?
Посчитайте долю отменённых заказов (в %) в **каждой** категории.
В какой категории отменяют чаще всего, в какой реже всего? Во сколько раз отличается?

Подсказка: цикл `for cat in df["category"].unique():`.

In [ ]:
shares = {}
for cat in df["category"].unique():
    part = df[df["category"] == cat]
    canc = part[part["status"] == CANCELLED]
    shares[cat] = len(canc) / len(part) * 100
    print(cat, ":", round(shares[cat], 1), "%")

top = max(shares, key=shares.get)
low = min(shares, key=shares.get)
print("Чаще всего отменяют:", top, "| реже всего:", low)
if shares[low] > 0:
    print("Разница в", round(shares[top] / shares[low], 1), "раза")
# Вывод: чаще всего отменяют смартфоны (12.2%), почти столько же мониторы (12.0%), реже всего планшеты (3.9%) — разница в 3.1 раза; со смартфонами и мониторами стоит разобраться, почему покупатели передумывают.

Смартфоны : 12.2 %
Аксессуары : 8.8 %
Планшеты : 3.9 %
Мониторы : 12.0 %
Ноутбуки : 6.2 %
Чаще всего отменяют: Смартфоны | реже всего: Планшеты
Разница в 3.1 раза


### Задача 7. Разведка `orders_raw.csv`
Загрузите сырой файл в переменную `raw`. Ничего не исправляйте, только считайте.
1. Сколько в нём строк и сколько разных `order_id`? Сколько лишних строк?
2. Сколько разных написаний города? Выпишите все варианты Алматы.
3. Сколько заказов найдёт `raw[raw["city"] == "Алматы"]`? А сколько заказов из Алматы в чистой таблице?

In [ ]:
raw = pd.read_csv("orders_raw.csv", dtype={"price": str})

print("Строк:", len(raw))
print("Разных order_id:", raw["order_id"].nunique())
print("Лишних строк:", len(raw) - raw["order_id"].nunique())

print("Разных написаний города:", raw["city"].nunique())
almaty = [c for c in raw["city"].dropna().unique()
          if "алма" in c.lower() or "alma" in c.lower()]
print("Варианты Алматы:", almaty)

print("raw == 'Алматы':", len(raw[raw["city"] == "Алматы"]))
print("Алматы в чистой таблице:", len(df[df["city"] == "Алматы"]))
# Вывод: в сыром файле 518 строк, но только 500 разных заказов — 18 строк-дублей; город записан 25 способами, у Алматы 5 вариантов (ALMATY, Almaty, алматы, с пробелами), поэтому фильтр == "Алматы" находит 86 заказов вместо 133 — теряется больше трети заказов, без чистки данных выводы будут неверными.

Строк: 518
Разных order_id: 500
Лишних строк: 18
Разных написаний города: 25
Варианты Алматы: ['Алматы', 'ALMATY', 'Almaty', 'алматы', ' Алматы ']
raw == 'Алматы': 86
Алматы в чистой таблице: 133


## Про

### Задача 8. Какие бывают «цены»
В `raw["price"]` записаны 518 значений. Разложите их по видам и посчитайте, сколько в каждом:

| Вид | Пример | Как найти |
|---|---|---|
| только цифры | `99000` | `raw["price"].str.isdigit() == True` |
| со знаком ₸ | `99000 ₸` | `.str.contains("₸", na=False)` |
| с `.0` на конце | `99000.0` | `.str.endswith(".0", na=False)` |
| с пробелом между цифрами | `99 000` | придумайте сами |
| слова вместо цены | ? | найдите через `value_counts()` или `unique()` |
| пропуск | `NaN` | `.isna()` |

Проверьте, что сумма всех видов — ровно 518. Если нет, какое-то значение попало в два вида или не попало никуда.

In [ ]:
s = raw["price"]

is_na  = s.isna()
digits = s.str.isdigit() == True
tenge  = s.str.contains("₸", na=False)
dot0   = s.str.endswith(".0", na=False)
space  = s.str.contains(" ", na=False) & (s.str.replace(" ", "").str.isdigit() == True)
words  = ~s.str.contains(r"\d", na=True)

kinds = {"только цифры": digits, "со знаком ₸": tenge, "с .0": dot0,
         "с пробелом": space, "слова": words, "пропуск": is_na}
total = 0
for name, mask in kinds.items():
    print(name, ":", mask.sum())
    total += mask.sum()
print("Сумма:", total, "из", len(s))

print("Слова вместо цены:", s[words].value_counts())

count = digits.astype(int) + tenge + dot0 + space + words + is_na
print("В двух видах:", s[count > 1].unique())
print("Никуда не попало:", s[count == 0].unique())
# Вывод: из 518 цен только 209 (около 40%) — чистые числа; остальные записаны со знаком ₸ (115), с .0 (92), с пробелом (92), словами «нет данных» и «abc» (8) или пропущены (2); сумма видов ровно 518, значит, каждое значение попало ровно в один вид — без очистки посчитать выручку по сырому файлу нельзя.

только цифры : 209
со знаком ₸ : 115
с .0 : 92
с пробелом : 92
слова : 8
пропуск : 2
Сумма: 518 из 518
Слова вместо цены: price
нет данных    4
abc           4
Name: count, dtype: int64
В двух видах: <StringArray>
[]
Length: 0, dtype: str
Никуда не попало: <StringArray>
[]
Length: 0, dtype: str


### Задача 9. Популярный ≠ прибыльный?
1. Какой товар **заказывают чаще всего** (по числу заказов)?
2. Какой товар принёс **больше всего выручки** по оплаченным заказам? Найдите его циклом по `df["product"].unique()`: считайте выручку каждого товара и запоминайте лучший.
3. Во сколько раз выручка самого прибыльного товара больше выручки самого популярного?

In [ ]:
popular = df["product"].value_counts().idxmax()
print("Чаще всего заказывают:", popular, "-", df["product"].value_counts().max(), "заказов")

best, best_rev = None, 0
for p in df["product"].unique():
    rev = paid[paid["product"] == p]["revenue"].sum()
    if rev > best_rev:
        best, best_rev = p, rev
print("Больше всего выручки:", best, "-", best_rev)

pop_rev = paid[paid["product"] == popular]["revenue"].sum()
print("Выручка самого популярного:", pop_rev)
print("Разница в", round(best_rev / pop_rev, 1), "раза")
# Вывод: чаще всего заказывают Xiaomi Redmi Note 13 (56 заказов), но больше всего выручки принёс MacBook Air M3 — 17 млн против 5.1 млн, в 3.3 раза больше; популярный товар не значит прибыльный: дорогой ноутбук даёт больше денег даже при меньшем числе заказов.

Чаще всего заказывают: Xiaomi Redmi Note 13 - 56 заказов
Больше всего выручки: MacBook Air M3 - 17019000
Выручка самого популярного: 5148000
Разница в 3.3 раза
